#### Transform Orders Data-Strings to JSON 
- Pre process the json string to fix data quality issues
- Transform JSON string to JSON object
- Write transformed data to silver layer

#### 1. Pre process the json string to fix data quality issues
- https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/regexp_replace

In [0]:
df = spark.read.table("gizmobox.bronze.py_orders")
display(df)

In [0]:
from pyspark.sql import functions as F
cleaned_df = df.select(
    F.regexp_replace(
            df.value, 
            '"order_date": (\\d{4}-\\d{2}-\\d{2})', 
            '"order_date": "$1"'
        ).alias('fixed_value')
)

display(cleaned_df)

####2. Transform JSON string to JSON object
- Function https://docs.databricks.com/aws/en/sql/language-manual/functions/schema_of_json
- Function https://docs.databricks.com/aws/en/sql/language-manual/functions/from_json
- https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.schema_of_json.html?highlight=schema_of_json
- https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.from_json.html?highlight=from_json


In [0]:
from pyspark.sql import functions as F

schema_df = cleaned_df.select(
    F.schema_of_json(cleaned_df.fixed_value.alias('schema'))
)

display(schema_df.limit(1))

In [0]:
schema = '''STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>'''

In [0]:
from pyspark.sql import functions as F

final_df = cleaned_df.select(
    F.from_json(F.col('fixed_value'), schema).alias('json_value')
)
display(final_df)

####3. Write transformed data to silver layer

In [0]:
final_df.writeTo("gizmobox.silver.py_orders_json").createOrReplace()

In [0]:
%sql
select * from gizmobox.silver.py_orders_json